# Lab 04 — Fine-Tuning Generative Models
**Session 04 · Desarrollo y Evaluación de Modelos Propios** · Especialización en IA Generativa y Desarrollo de Negocios · Universidad del Rosario · Rosario GSB

[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/04-fine-tuning/lab.ipynb)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/04-fine-tuning/lab.ipynb)

In the slides we compared **full fine-tuning** (every weight changes) with **parameter-efficient fine-tuning** (a few new weights change). Today you do both:

1. **Full fine-tuning** of an encoder, `distilbert/distilbert-base-uncased`, to route Andes Bank customer messages into 10 intents from **Banking77**, using the Hugging Face `Trainer`.
2. **LoRA** supervised fine-tuning (SFT) of a small chat model, `HuggingFaceTB/SmolLM2-135M-Instruct`, so that it answers in the **brand voice** of Andes Bank (a fictional digital bank), using 40 instruction → response pairs.

### Learning objectives
- Prepare a labelled dataset: subset, stratified validation split, a held-out test set and a simple baseline.
- Fine-tune DistilBERT with `Trainer`, read its learning curves and report accuracy and macro-F1 on the test set **once**.
- Format chat data with a chat template, mask the prompt tokens and train a LoRA adapter with a plain PyTorch loop.
- Compare outputs before and after fine-tuning, and check for side effects (forgetting).

### Time plan (60 min)
| Part | Topic | Time |
|---|---|---|
| 0 | Setup and Kaggle checklist | 3' |
| 1 | Banking77: 10 intents, stratified split, TF-IDF baseline | 12' |
| 2 | Full fine-tuning of DistilBERT with `Trainer` | 23' |
| 3 | LoRA brand voice for SmolLM2-135M-Instruct | 20' |
| — | Wrap-up | 2' |

### ✅ Kaggle checklist
- **Settings → Accelerator → GPU T4 x2** (the notebook also runs on CPU, but training is several times slower).
- **Settings → Internet → On** (models and data are downloaded from the Hugging Face Hub).
- Run the cells **in order**, from the top. If you change the accelerator, the session restarts: run everything again.

## Part 0 — Setup
The first cell is the same in every lab: it fixes the random seeds, detects the GPU and defines `FAST_DEV_RUN`
(used by the course's automatic tests to run a tiny version of the notebook; leave it alone).

In [ ]:
# --- Course setup (do not edit) ---
import os, sys, random, subprocess, importlib
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"  # used by the course smoke test
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules

def ensure(package, import_name=None):
    """Install a package only if it is missing (Kaggle already ships most of them)."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import socket
if os.environ.get("HF_HUB_OFFLINE") != "1":  # models and datasets come from the Hugging Face Hub
    try:
        socket.create_connection(("huggingface.co", 443), timeout=10).close()
    except OSError:
        raise RuntimeError(
            "No internet connection: this notebook downloads models and datasets from Hugging Face.\n"
            "Kaggle: in the right-hand panel open Settings (Session options) and switch Internet on. "
            "The switch only appears once your account is phone-verified (kaggle.com/settings). "
            "Then run this cell again. Colab has internet by default. "
            "On your own computer: check your connection, or set HF_HUB_OFFLINE=1 if the models are already cached."
        ) from None

import numpy as np
import torch
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE} | Kaggle: {IN_KAGGLE} | Colab: {IN_COLAB} | Fast dev run: {FAST_DEV_RUN}")
# --- end of course setup ---

# Packages for this session
ensure("transformers")
ensure("datasets")
ensure("accelerate")
ensure("peft")
ensure("scikit-learn", "sklearn")

In [ ]:
import json, math, time, textwrap, urllib.request
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import transformers, datasets, peft

pd.set_option("display.max_colwidth", 120)
print("transformers", transformers.__version__, "| datasets", datasets.__version__, "| peft", peft.__version__)
if DEVICE == "cpu":
    print("⚠️ No GPU detected: everything works, but training will be slower. On Kaggle: Settings → Accelerator → GPU T4 x2.")

## Part 1 — Banking77: from 77 intents to a 10-intent routing task
**Banking77** (Casanueva et al., 2020) contains 13,083 real-style customer queries from a single banking domain, each labelled with one of **77 fine-grained intents**.
We use the Hub copy `legacy-datasets/banking77`, which has a `train` and a `test` split. The label names live in the dataset's features.

In [ ]:
from datasets import load_dataset, ClassLabel

raw = load_dataset("legacy-datasets/banking77")
ALL_INTENTS = raw["train"].features["label"].names
print(raw)
print(f"{len(ALL_INTENTS)} intents, for example: {ALL_INTENTS[:6]}")
print("Example:", raw["train"][0], "→", ALL_INTENTS[raw["train"][0]["label"]])

Andes Bank wants to route messages for **10 intents** first. We chose them on purpose: two pairs are easy to confuse
(`card_arrival` vs `card_delivery_estimate`, `exchange_rate` vs `card_payment_wrong_exchange_rate`), as on the flip cards in the slides.

We keep only those intents and **renumber the labels 0–9**, so the classifier has exactly 10 outputs.

In [ ]:
INTENTS = [
    "activate_my_card", "card_arrival", "card_delivery_estimate", "lost_or_stolen_card",
    "exchange_rate", "card_payment_wrong_exchange_rate", "transfer_timing", "top_up_failed",
    "pin_blocked", "terminate_account",
]
OLD_TO_NEW = {ALL_INTENTS.index(name): new_id for new_id, name in enumerate(INTENTS)}

def keep_intents(split):
    """Rows of `split` whose intent is in INTENTS, relabelled 0..len(INTENTS)-1."""
    ds = raw[split].filter(lambda ex: ex["label"] in OLD_TO_NEW)
    features = ds.features.copy()
    features["label"] = ClassLabel(names=INTENTS)
    return ds.map(lambda ex: {"label": OLD_TO_NEW[ex["label"]]}, features=features)

train_full = keep_intents("train")
test_ds = keep_intents("test")
print(f"train: {len(train_full)} rows · test: {len(test_ds)} rows")

Banking77 has **no validation split**. We carve one out of `train` with a **stratified** split (every intent keeps the same proportion in both parts).
The official `test` split stays untouched until the very end: we will use it **once**.

In fast-dev mode (automatic tests only) we keep a handful of examples per intent; the code path is identical.

In [ ]:
def take_per_class(ds, n, seed=SEED):
    """Stratified subsample: at most `n` rows per label (all rows when n is None)."""
    if n is None:
        return ds
    rows_by_label = {}
    for i, y in enumerate(list(ds["label"])):
        rows_by_label.setdefault(y, []).append(i)
    rnd = random.Random(seed)
    keep = []
    for y in sorted(rows_by_label):
        idx = rows_by_label[y]
        rnd.shuffle(idx)
        keep += idx[:n]
    return ds.select(sorted(keep))

split = train_full.train_test_split(test_size=0.15, stratify_by_column="label", seed=SEED)
train_ds = take_per_class(split["train"], 15 if FAST_DEV_RUN else None)
val_ds = take_per_class(split["test"], 5 if FAST_DEV_RUN else None)
test_ds = take_per_class(test_ds, 10 if FAST_DEV_RUN else None)

counts = pd.DataFrame({
    "train": pd.Series(list(train_ds["label"])).value_counts().sort_index(),
    "validation": pd.Series(list(val_ds["label"])).value_counts().sort_index(),
    "test": pd.Series(list(test_ds["label"])).value_counts().sort_index(),
})
counts.index = INTENTS
counts.loc["TOTAL"] = counts.sum()
counts

In [ ]:
# Two examples per intent
for label_id, name in enumerate(INTENTS):
    examples = [t for t, y in zip(list(train_ds["text"]), list(train_ds["label"])) if y == label_id][:2]
    print(f"{name:35s} | " + " || ".join(examples))

### ✋ Checkpoint (Zoom chat)
1. Which **pair** of intents do you expect the model to confuse most? Why?
2. The classes are not balanced (see the table). Why do we report **macro-F1** in addition to accuracy?

### A baseline first: TF-IDF + logistic regression
Before training a Transformer, measure what a simple, cheap model achieves. If fine-tuning cannot beat this baseline, it is not worth its cost.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

def scores(y_true, y_pred):
    return {"accuracy": accuracy_score(y_true, y_pred),
            "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0)}

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True)
X_train = vectorizer.fit_transform(list(train_ds["text"]))
baseline = LogisticRegression(max_iter=2000).fit(X_train, list(train_ds["label"]))
baseline_val = scores(list(val_ds["label"]), baseline.predict(vectorizer.transform(list(val_ds["text"]))))
print("TF-IDF + logistic regression (validation):", {k: round(v, 4) for k, v in baseline_val.items()})

### 🧪 Try it
Change `ngram_range=(1, 2)` to `(1, 1)` (single words only) and re-run the cell. Does the baseline get better or worse on validation?

<details><summary>Show a solution</summary>

Word pairs such as "exchange rate" or "not arrived" carry intent information that single words miss, so `(1, 2)` is usually a little better.
The exact difference depends on the split: with a validation set of about 190 messages, one message is about 0.5 accuracy points, so small differences can be noise.
</details>

## Part 2 — Full fine-tuning of DistilBERT with `Trainer`
### 2.1 Tokenize, after checking the lengths
`max_length` truncates long messages. Choose it from the data: cover about 95% of the examples, because compute and memory grow with sequence length.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = np.array([len(ids) for ids in tokenizer(list(train_ds["text"]))["input_ids"]])
print(f"Tokens per message: median {np.median(lengths):.0f} · 95th percentile {np.percentile(lengths, 95):.0f} · max {lengths.max()}")

MAX_LENGTH = 32 if FAST_DEV_RUN else 64

def tokenize(batch):
    # No padding here: DataCollatorWithPadding pads each batch to its own longest message.
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_tok = train_ds.map(tokenize, batched=True)
val_tok = val_ds.map(tokenize, batched=True)
test_tok = test_ds.map(tokenize, batched=True)
print(train_tok)

### 2.2 The model: a pretrained encoder plus a new classification head
`AutoModelForSequenceClassification` loads DistilBERT's pretrained weights and adds a **new, randomly initialised** head with 10 outputs.
The loading report lists the head's weights as *missing* (they are new) and the language-modelling head as *unexpected* (we do not need it): that is expected.

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = dict(enumerate(INTENTS))
label2id = {name: i for i, name in id2label.items()}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(INTENTS), id2label=id2label, label2id=label2id)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parameters: {n_total:,} · trainable: {n_trainable:,} ({100 * n_trainable / n_total:.0f}%) → full fine-tuning")

### 2.3 Metrics
`Trainer` calls `compute_metrics` after each evaluation with the logits and the true labels.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return scores(labels, preds)

### 2.4 Training arguments and training
Each argument corresponds to a knob from the slides:

| Argument | Meaning |
|---|---|
| `learning_rate=5e-5` | inside BERT's recommended grid {2e-5, 3e-5, 5e-5} |
| `num_train_epochs` | passes over the training set (3 here) |
| `per_device_train_batch_size` | examples per update and per GPU |
| `warmup_steps` | the first ≈10% of updates ramp the learning rate up from 0 |
| `weight_decay=0.01` | mild regularisation (AdamW) |
| `eval_strategy="epoch"` | evaluate on **validation** after every epoch, so we can see overfitting |
| `save_strategy="no"`, `report_to="none"` | no checkpoints on disk and no external loggers: keeps the lab light |

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding

LEARNING_RATE = 5e-5
EPOCHS = 1 if FAST_DEV_RUN else 3
BATCH_SIZE = 16 if FAST_DEV_RUN else 32
total_steps = math.ceil(len(train_tok) / BATCH_SIZE) * EPOCHS

args = TrainingArguments(
    output_dir="outputs/distilbert-banking77",
    learning_rate=LEARNING_RATE,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=64,
    warmup_steps=int(0.1 * total_steps),
    weight_decay=0.01,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="no",
    report_to="none",
    seed=SEED,
    use_cpu=(DEVICE == "cpu"),  # keep Trainer on the same device as the rest of the notebook
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
train_seconds = time.time() - start
print(f"Training time: {train_seconds:.0f} s on {DEVICE}")

### 2.5 Learning curves
`trainer.state.log_history` keeps every log line: training loss per epoch and validation metrics per epoch.

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
train_curve = history.dropna(subset=["loss"])[["epoch", "loss"]] if "loss" in history else pd.DataFrame(columns=["epoch", "loss"])
val_curve = history.dropna(subset=["eval_loss"])[["epoch", "eval_loss", "eval_accuracy", "eval_f1_macro"]]
display(val_curve.round(4))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(train_curve["epoch"], train_curve["loss"], "o-", label="training loss")
axes[0].plot(val_curve["epoch"], val_curve["eval_loss"], "s--", label="validation loss")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss"); axes[0].legend(); axes[0].set_title("Loss")
axes[1].plot(val_curve["epoch"], val_curve["eval_accuracy"], "o-", label="accuracy")
axes[1].plot(val_curve["epoch"], val_curve["eval_f1_macro"], "s--", label="macro-F1")
axes[1].set_xlabel("epoch"); axes[1].set_ylim(0, 1.02); axes[1].legend(); axes[1].set_title("Validation metrics")
plt.tight_layout(); plt.show()

### ✋ Checkpoint
1. Is there any sign of **overfitting** (training loss falling while validation loss rises)? Which epoch would you keep?
2. The first training loss is close to ln(10) ≈ 2.3 in a fresh run. Why that number?

### 2.6 The test set, once
We chose everything (max length, learning rate, epochs) with validation. Now we measure the final model on the untouched **test** split **one time**, and compare with the baseline on the same split.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(test_tok, metric_key_prefix="test")
y_true, y_pred = pred_out.label_ids, pred_out.predictions.argmax(-1)
finetuned_test = scores(y_true, y_pred)
baseline_test = scores(list(test_ds["label"]), baseline.predict(vectorizer.transform(list(test_ds["text"]))))
finetuned_val = {"accuracy": float(val_curve["eval_accuracy"].iloc[-1]), "f1_macro": float(val_curve["eval_f1_macro"].iloc[-1])}

comparison = pd.DataFrame({
    "val accuracy": [baseline_val["accuracy"], finetuned_val["accuracy"]],
    "val macro-F1": [baseline_val["f1_macro"], finetuned_val["f1_macro"]],
    "test accuracy": [baseline_test["accuracy"], finetuned_test["accuracy"]],
    "test macro-F1": [baseline_test["f1_macro"], finetuned_test["f1_macro"]],
}, index=["TF-IDF + logistic regression", "DistilBERT fine-tuned"]).round(4)
display(comparison)
print(classification_report(y_true, y_pred, labels=list(range(len(INTENTS))), target_names=INTENTS, digits=3, zero_division=0))

In [ ]:
# Which intents get mixed up? The largest off-diagonal cells of the confusion matrix.
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(INTENTS))))
pairs = [(cm[i, j], INTENTS[i], INTENTS[j]) for i in range(len(INTENTS)) for j in range(len(INTENTS)) if i != j and cm[i, j] > 0]
top_confusions = pd.DataFrame(sorted(pairs, reverse=True)[:5], columns=["count", "true intent", "predicted as"])
display(top_confusions if len(top_confusions) else "No confusions on this test set.")

### 2.7 Route new messages
The model outputs one probability per intent. We show the top two and the confidence.
Note the last message: it is **out of scope** (none of the 10 intents), but a classifier always picks one of its classes. In Session 06 you will add a confidence threshold that escalates such cases to a person.

In [ ]:
def route(messages, top_k=2):
    model.eval()
    enc = tokenizer(messages, truncation=True, max_length=MAX_LENGTH, padding=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        probs = torch.softmax(model(**enc).logits, dim=-1).cpu().numpy()
    rows = []
    for msg, p in zip(messages, probs):
        best = np.argsort(p)[::-1][:top_k]
        rows.append({"message": msg,
                     "intent": INTENTS[best[0]], "confidence": round(float(p[best[0]]), 3),
                     "runner-up": f"{INTENTS[best[1]]} ({p[best[1]]:.3f})"})
    return pd.DataFrame(rows)

new_messages = [
    "Hi, I ordered my card 10 days ago and it's still not here",
    "my pin got blocked after 3 wrong tries, help",
    "The rate you charged for my purchase in dollars looks off",
    "I want to shut down my account for good",
    "How long will it take for money to reach my friend's account?",
    "Can I get a mortgage with you?",   # out of scope
]
route(new_messages)

### 🧪 Try it
Set `LEARNING_RATE = 1e-3` in the training cell (2.4), re-run cells 2.2 → 2.6 and compare the curves and the test accuracy.

<details><summary>Show a solution</summary>

`1e-3` is 20× the recommended value, and full fine-tuning becomes **unstable**. Depending on the seed, batch size and warmup, validation accuracy may stall on a plateau for an epoch or two, the loss may jump around, or training may diverge completely (loss stuck near ln(10) ≈ 2.3, one intent predicted for everything).
In our test runs it usually still learned something, but more slowly and with worse final scores than `5e-5`: large updates damage the pretrained features in the first steps. The challenge shows the fully diverged shape with an even larger learning rate.
Remember to set it back to `5e-5` (and re-run 2.2 → 2.6) before Part 3.
</details>

## Part 3 — LoRA: teach SmolLM2 the Andes Bank brand voice
Andes Bank's brand guide says: **warm, concise, concrete**. Answers open with a short empathetic line, give one or two steps in the Andes app,
never ask for PINs or passwords, and close with the sign-off *"— Andes Bank, here for you."*

The file `data/brand_voice.jsonl` has 40 examples written for this course (Andes Bank is fictional). One JSON object per line:
`{"instruction": ..., "response": ...}`. We train on the first 32 and keep 8 for validation; the last 3 are our **held-out demo prompts**.

In [ ]:
DATA_URL = ("https://raw.githubusercontent.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/"
            "main/sessions/04-fine-tuning/data/brand_voice.jsonl")
local_file = Path("data/brand_voice.jsonl")
if local_file.exists():
    jsonl_text = local_file.read_text(encoding="utf-8")
else:  # Kaggle and Colab open the notebook without the repository's data folder
    jsonl_text = urllib.request.urlopen(DATA_URL, timeout=30).read().decode("utf-8")
rows = [json.loads(line) for line in jsonl_text.splitlines() if line.strip()]

train_rows, val_rows = rows[:32], rows[32:]
if FAST_DEV_RUN:
    train_rows, val_rows = train_rows[:8], val_rows[-4:]
DEMO_ROWS = rows[-3:]   # held out: never used for training
print(f"{len(rows)} examples · train {len(train_rows)} · validation {len(val_rows)}")
print(json.dumps(rows[3], indent=2, ensure_ascii=False))

### 3.1 Load the chat model and look at its chat template
We load the model in float32, which trains stably on any GPU or CPU (135 M parameters × 4 bytes ≈ 0.5 GB).
Every example becomes a conversation: a **system** message (the same at training and inference time), the customer's **user** message and the **assistant**'s answer.

In [ ]:
from transformers import AutoModelForCausalLM

LM_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
lm_tokenizer = AutoTokenizer.from_pretrained(LM_NAME)
lm = AutoModelForCausalLM.from_pretrained(LM_NAME).float().to(DEVICE)
SYSTEM_PROMPT = "You are the virtual assistant of Andes Bank, a digital bank."

def to_messages(instruction, response=None):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": instruction}]
    if response is not None:
        messages.append({"role": "assistant", "content": response})
    return messages

print(lm_tokenizer.apply_chat_template(to_messages(rows[3]["instruction"], rows[3]["response"]), tokenize=False))

### 3.2 Before fine-tuning
Greedy decoding (`do_sample=False`) makes the comparison fair: same prompt, same decoding, only the weights will change.
A mild `repetition_penalty` stops the small model from looping.
We also measure three simple **brand-voice indicators** on the validation prompts: average length, sign-off rate and how often the answer points to the Andes app.

In [ ]:
MAX_NEW_TOKENS = 24 if FAST_DEV_RUN else 96
SIGN_OFF = "Andes Bank, here for you"

@torch.no_grad()
def reply(model, instruction):
    model.eval()
    inputs = lm_tokenizer.apply_chat_template(to_messages(instruction), add_generation_prompt=True,
                                              return_tensors="pt", return_dict=True).to(DEVICE)
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                         repetition_penalty=1.1, pad_token_id=lm_tokenizer.eos_token_id)
    return lm_tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def style_indicators(texts):
    return {"avg_words": float(np.mean([len(t.split()) for t in texts])),
            "sign_off_rate": float(np.mean([SIGN_OFF in t for t in texts])),
            "mentions_app": float(np.mean(["app" in t.lower() for t in texts]))}

start = time.time()
before = {r["instruction"]: reply(lm, r["instruction"]) for r in val_rows}
style_before = style_indicators(list(before.values()))
print(f"Generated {len(before)} answers in {time.time() - start:.0f} s")
for r in DEMO_ROWS:
    print(f"\n👤 {r['instruction']}\n🤖 BEFORE: {textwrap.fill(before[r['instruction']], 110)}")

### 3.3 Tokenize for SFT, with the prompt masked
We train the model to **write the answer**, not to predict the system and user text. Tokens of the prompt get label `-100`, which the loss ignores.
The end-of-turn token `<|im_end|>` stays in the labels: that is how the model learns to stop.

In [ ]:
def encode_sft(row):
    prompt_ids = lm_tokenizer.apply_chat_template(to_messages(row["instruction"]), add_generation_prompt=True,
                                                  tokenize=True, return_dict=True)["input_ids"]
    full_ids = lm_tokenizer.apply_chat_template(to_messages(row["instruction"], row["response"]),
                                                tokenize=True, return_dict=True)["input_ids"]
    assert full_ids[:len(prompt_ids)] == prompt_ids, "the chat template must extend the prompt"
    return {"input_ids": full_ids, "labels": [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]}

def collate(batch):
    """Right-pad a list of encoded examples into tensors (padding is masked in the loss too)."""
    width = max(len(b["input_ids"]) for b in batch)
    input_ids = torch.full((len(batch), width), lm_tokenizer.pad_token_id)
    labels = torch.full((len(batch), width), -100)
    attention_mask = torch.zeros((len(batch), width), dtype=torch.long)
    for i, b in enumerate(batch):
        n = len(b["input_ids"])
        input_ids[i, :n] = torch.tensor(b["input_ids"])
        labels[i, :n] = torch.tensor(b["labels"])
        attention_mask[i, :n] = 1
    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_enc = [encode_sft(r) for r in train_rows]
val_enc = [encode_sft(r) for r in val_rows]

example = train_enc[0]
first_trained = next(i for i, y in enumerate(example["labels"]) if y != -100)
window = range(first_trained - 6, first_trained + 8)
display(pd.DataFrame({"token": [lm_tokenizer.decode([example["input_ids"][i]]) for i in window],
                      "in the loss?": ["yes" if example["labels"][i] != -100 else "no (masked)" for i in window]}).T)
print(f"Example 0: {len(example['input_ids'])} tokens, of which {sum(y != -100 for y in example['labels'])} are trained on")

### 3.4 Add LoRA adapters
Same configuration as in the slides: rank 8, α = 16 (scale α/r = 2), dropout 0.05, on the **query** and **value** projections of every layer.
Check the printed count against the LoRA calculator: 30 layers × 8 × [(576 + 576) + (576 + 192)] = 460,800.

In [ ]:
from peft import LoraConfig, get_peft_model

print("Linear layers in one block:", sorted({n.split(".")[-1] for n, m in lm.named_modules() if isinstance(m, torch.nn.Linear)}))
lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM")
lora_model = get_peft_model(lm, lora_config)
lora_model.print_trainable_parameters()

cfg = lm.config
kv_dim = cfg.num_key_value_heads * (cfg.hidden_size // cfg.num_attention_heads)
by_formula = cfg.num_hidden_layers * lora_config.r * ((cfg.hidden_size + cfg.hidden_size) + (cfg.hidden_size + kv_dim))
print(f"By the formula: {by_formula:,} trainable weights (hidden {cfg.hidden_size}, k/v width {kv_dim}, {cfg.num_hidden_layers} layers)")

### 3.5 A plain PyTorch training loop
No magic: shuffle, batch, forward, `loss.backward()`, optimizer step. LoRA uses a higher learning rate than full fine-tuning (few weights move, and `B` starts at zero).
We compute the loss on the validation examples before training and after every epoch.

In [ ]:
from transformers import get_linear_schedule_with_warmup

LORA_LR = 1e-3
LORA_EPOCHS = 2 if FAST_DEV_RUN else 8
LORA_BATCH = 4

@torch.no_grad()
def average_loss(model, encoded):
    model.eval()
    total, count = 0.0, 0
    for i in range(0, len(encoded), LORA_BATCH):
        batch = {k: v.to(DEVICE) for k, v in collate(encoded[i:i + LORA_BATCH]).items()}
        n_tokens = (batch["labels"][:, 1:] != -100).sum().item()   # tokens that count in the loss
        total += model(**batch).loss.item() * n_tokens
        count += n_tokens
    return total / count

trainable = [p for p in lora_model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LORA_LR, weight_decay=0.0)
steps = math.ceil(len(train_enc) / LORA_BATCH) * LORA_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * steps), num_training_steps=steps)
shuffler = torch.Generator().manual_seed(SEED)

lora_log = [{"epoch": 0, "train_loss": None, "val_loss": average_loss(lora_model, val_enc)}]
start = time.time()
for epoch in range(1, LORA_EPOCHS + 1):
    lora_model.train()
    order = torch.randperm(len(train_enc), generator=shuffler).tolist()
    epoch_losses = []
    for i in range(0, len(order), LORA_BATCH):
        batch = {k: v.to(DEVICE) for k, v in collate([train_enc[j] for j in order[i:i + LORA_BATCH]]).items()}
        loss = lora_model(**batch).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, max_norm=1.0)
        optimizer.step(); scheduler.step(); optimizer.zero_grad()
        epoch_losses.append(loss.item())
    lora_log.append({"epoch": epoch, "train_loss": float(np.mean(epoch_losses)), "val_loss": average_loss(lora_model, val_enc)})
    print(f"epoch {epoch}: train loss {lora_log[-1]['train_loss']:.3f} · validation loss {lora_log[-1]['val_loss']:.3f}")
lora_seconds = time.time() - start
print(f"LoRA training: {steps} steps in {lora_seconds:.0f} s on {DEVICE}")

log_df = pd.DataFrame(lora_log)
plt.figure(figsize=(6, 3.4))
plt.plot(log_df["epoch"][1:], log_df["train_loss"][1:], "o-", label="training loss")
plt.plot(log_df["epoch"], log_df["val_loss"], "s--", label="validation loss")
plt.xlabel("epoch"); plt.ylabel("loss (answer tokens)"); plt.legend(); plt.title("LoRA SFT on 32 brand-voice examples")
plt.tight_layout(); plt.show()

### 3.6 After fine-tuning: same prompts, same decoding

In [ ]:
after = {r["instruction"]: reply(lora_model, r["instruction"]) for r in val_rows}
style_after = style_indicators(list(after.values()))

for r in DEMO_ROWS:
    q = r["instruction"]
    print(f"\n👤 {q}\n🤖 BEFORE: {textwrap.fill(before[q], 110)}\n🟢 AFTER:  {textwrap.fill(after[q], 110)}")

display(pd.DataFrame({"before": style_before, "after LoRA": style_after}).T.round(3))

### 3.7 Side effects and the frozen base
Two checks from the slides on catastrophic forgetting:
1. **Switch the adapter off** (`disable_adapter`) and the original model comes back: its weights were never changed.
2. Ask a **general question** outside the bank's domain. Does the adapter leak the brand voice (or bank topics) into unrelated answers?

Finally we save the adapter and look at its size on disk.

In [ ]:
demo_q = DEMO_ROWS[0]["instruction"]
with lora_model.disable_adapter():
    base_again = reply(lora_model, demo_q)
print("Adapter off reproduces the original answer:", base_again == before[demo_q])

general_q = "Give me one tip to sleep better."
with lora_model.disable_adapter():
    general_base = reply(lora_model, general_q)
general_lora = reply(lora_model, general_q)
print(f"\n👤 {general_q}\n🤖 BASE: {textwrap.fill(general_base, 110)}\n🟢 LORA: {textwrap.fill(general_lora, 110)}")

ADAPTER_DIR = Path("outputs/andes-brand-voice-lora")
lora_model.save_pretrained(ADAPTER_DIR)
adapter_mb = sum(f.stat().st_size for f in ADAPTER_DIR.rglob("*") if f.is_file()) / 1e6
print(f"\nAdapter saved in {ADAPTER_DIR}: {adapter_mb:.1f} MB on disk "
      f"(the full model in float32 would be ≈ {4 * sum(p.numel() for p in lm.base_model.parameters()) / 1e6:.0f} MB)")

### ✋ Checkpoint
1. Name **two** concrete style changes between BEFORE and AFTER. Did the model learn **facts** about Andes Bank, or a **style**? How do you know?
2. The adapter trains about 0.34% of the weights. What does that tell you about how much of the model has to change to alter its style?
3. Would you deploy this 135 M model to real customers? What would you evaluate first (Session 05)?

### 🧪 Try it
Change `target_modules` to `["q_proj", "k_proj", "v_proj", "o_proj"]` or the rank to `r=2`, then re-run from 3.1 (the model must be reloaded before adding new adapters).
How many weights train now? Compare with the LoRA calculator in the slides.

<details><summary>Show a solution</summary>

Per layer, each targeted matrix adds r × (inputs + outputs) weights. With q, k, v, o and r = 8:
8 × [(576 + 576) + (576 + 192) + (576 + 192) + (576 + 576)] = 30,720 per layer, × 30 layers = **921,600** (≈0.68%).
With only q, v and r = 2: 30 × 2 × 1,920 = **115,200**. Fewer weights train faster and overfit less, but may not capture the style as well:
compare the validation loss and the sign-off rate.
</details>

## Wrap-up
- **Full fine-tuning** changed all 67 M weights of DistilBERT and beat a TF-IDF baseline on 10 fine-grained intents; we chose settings with **validation** and reported **test** once.
- **LoRA** changed about 0.34% of SmolLM2's weights, enough to shift its style towards the brand voice; the base model is intact and the adapter is a small file.
- Curves tell you what to do next: underfitting, a good fit, overfitting or divergence.
- Style is learnable from a few dozen examples; **facts that change** (fees, dates) belong in RAG, not in the weights.

**Next: the Session Challenge.** You will fine-tune DistilBERT on **your own 10 intents** with two training configurations, compare their curves,
decide between fine-tuning, RAG and prompting for a business case, and critique training settings proposed by an AI assistant.